# Parallelism

## Threading

Python provides a threading library that allows the creation of multiple threads that executes within a Python program. It is a bit of code overhead. But we could represent a multithreaded for-loop execution in the following way:

In [1]:
import numpy as np
import threading
import multiprocessing

def worker(arr1, arr2, arr3, chunk):
    """The thread worker."""

    for index in chunk:
        arr3[index] = arr1[index] + arr2[index]

nthreads = multiprocessing.cpu_count()

n = 1000000
a = np.random.randn(n)
b = np.random.randn(n)

c = np.empty(n, dtype='float64')


def run_with_n_threads(nthreads):
    chunks = np.array_split(range(n), nthreads)
    all_threads = []
    for chunk in chunks:
        thread = threading.Thread(target=worker, args=(a, b, c, chunk))
        all_threads.append(thread)
        thread.start()

    for thread in all_threads:
        thread.join()

### Run with 1 thread

Lets run with just 1 thread to see how much time it takes.


In [6]:
%timeit run_with_n_threads(1)

569 ms ± 24 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Run with 2 threads

Now lets run with 2 threads to see how much time it takes.

In [7]:
%timeit run_with_n_threads(2)

577 ms ± 19.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Run with 4 threads

Okay strange, lets run 4 threads!

In [8]:
%timeit run_with_n_threads(4)

547 ms ± 1.86 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


WHAT IS GOING ON? Back to the slides!

## Multiprocessing

Instead of multiple threads we use multiple Python processes, each with its own GIL and memory space.

In [ ]:
multiprocessing.set_start_method('fork')

In [13]:
import ctypes

 # Only needed for Jupyter notebooks

def worker(arr1, arr2, arr3, chunk):
    """The thread worker."""

    # Create Numpy arrays from the
    # shared multiprocessing arrays

    arr1_np = np.frombuffer(arr1.get_obj())
    arr2_np = np.frombuffer(arr2.get_obj())
    arr3_np = np.frombuffer(arr3.get_obj())

    for index in chunk:
        arr3_np[index] = arr1_np[index] + arr2_np[index]

nprocesses = multiprocessing.cpu_count()

n = 1000000

a = multiprocessing.Array(ctypes.c_double, n)
b = multiprocessing.Array(ctypes.c_double, n)
c = multiprocessing.Array(ctypes.c_double, n)


a[:] = np.random.randn(n)
b[:] = np.random.randn(n)



def run_with_n_processes(nprocesses):
    all_processes = []
    chunks = np.array_split(range(n), nprocesses)
    for chunk in chunks:
        process = multiprocessing.Process(target=worker, args=(a, b, c, chunk))
        all_processes.append(process)
        process.start()

    for process in all_processes:
        process.join()

In [14]:
%timeit run_with_n_processes(1)

689 ms ± 83.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
%timeit run_with_n_processes(2)

446 ms ± 69.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
%timeit run_with_n_processes(4)

271 ms ± 9.54 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Ok we are getting somewhere